# Reproduce a Known SAE Result

**Goal:** Prove the tooling works end-to-end before touching abstention. A factual prompt is run through Gemma 3 1B Instruct and a Gemma Scope 2 SAE, and top-activating features are verified to be interpretable through Neuronpedia. SAE reconstruction quality is also measured.

This notebook mirrors `scripts/00_reproduce_known.py` with interactive exploration.

## Setup

Load the model (Gemma 3 1B Instruct, ~2 GB in fp16) and the SAE (Gemma Scope 2, layer 13, 16k width) onto the GPU.

In [5]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json
from pathlib import Path
import torch
from src.config import setup
from src.model import load_model, load_sae, print_memory_report

config = setup(None)
model = load_model(config)
sae = load_sae(config, layer=config["sae"]["primary_layer"])
print_memory_report()

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 805.51it/s]



== Model Info ==
Model loaded: google/gemma-3-1b-it
Parameters: 1000M
VRAM used: 4.23GB


== SAE Info ==
SAE loaded: layer_13_width_16k_l0_medium
Dictionary size: 16384
Input dimension: 1152
VRAM used: 4.31GB


VRAM: 4.31GB allocated / 4.39GB reserved / 8.59GB total


## A. Factual Completion Prompt

Run a simple factual prompt through the model with the SAE hooked in. The model should predict "Paris".

In [6]:
prompt = "The Eiffel Tower is in the city of"
tokens = model.to_tokens(prompt)
print(f"Prompt: '{prompt}'")
print(f"Tokens shape: {tokens.shape} (batch, seq)")

logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])

top_token = logits[0, -1].argmax()
print(f"Top predicted token: '{model.tokenizer.decode(top_token)}'")

Prompt: 'The Eiffel Tower is in the city of'
Tokens shape: torch.Size([1, 9]) (batch, seq)
Top predicted token: ' Paris'


## B. Extract SAE Feature Activations

Extract the SAE feature activations at the last token position (where the next-token prediction happens). Between 50 and200 active features are expected out of 16,384 (sparse!), and the top features should relate to geography, cities and France.

In [7]:
layer = config["sae"]["primary_layer"]
hook_name = model.get_sae_hook_name(sae, internal="hook_sae_acts_post")
sae_acts = cache[hook_name]  # shape: [1, seq_len, n_features]
print(f"SAE activations shape: {sae_acts.shape}")

last_token_acts = sae_acts[0, -1, :]  # shape: [n_features]

n_active = (last_token_acts > 0).sum().item()
print(f"Active features at last token: {n_active} / {last_token_acts.shape[0]}")
print(f"Sparsity (L0): {n_active}")

SAE activations shape: torch.Size([1, 9, 16384])
Active features at last token: 70 / 16384
Sparsity (L0): 70


In [8]:
top_k = 20
top_values, top_indices = last_token_acts.topk(top_k)

print(
    f"Top-{top_k} features at last token ('{model.tokenizer.decode(tokens[0, -1])}'):"
)
print(f"{'Rank':<6} {'Feature ID':<12} {'Activation':<12}")
print("-" * 30)
for rank, (index, value) in enumerate(zip(top_indices, top_values)):
    print(f"{rank + 1:<6} {index.item():<12} {value.item():<12.4f}")

Top-20 features at last token (' of'):
Rank   Feature ID   Activation  
------------------------------
1      208          289.7500    
2      667          283.7500    
3      210          266.2500    
4      7055         210.2500    
5      374          199.0000    
6      1553         181.7500    
7      249          151.1250    
8      63           139.6250    
9      485          120.0000    
10     90           115.5625    
11     6639         114.0000    
12     1732         105.1250    
13     613          104.5625    
14     1961         98.0625     
15     1110         97.5625     
16     6530         96.1875     
17     272          95.1250     
18     635          94.1250     
19     710          90.3750     
20     81           88.8750     


## C. Neuronpedia Lookup

For each top feature, look it up on Neuronpedia to verify it has an interpretable meaning.

**URL pattern:**
```
https://www.neuronpedia.org/gemma-3-1b-it/13-gemmascope-2-res-16k/<FEATURE_ID>
```

`<FEATURE_ID>` is replaced with the feature index from the table above. The results are saved in `results/00/top_features.csv`.

**Expected:** The top features should relate to geography, cities, France, landmarks or similar concepts relevant to the Eiffel Tower prompt.

In [10]:
import pandas as pd
from src.neuronpedia import fetch_neuronpedia_explanations

model_id = "gemma-3-1b-it"
sae_id = f"{layer}-gemmascope-2-res-16k"
feature_ids = [idx.item() for idx in top_indices]
activations = [val.item() for val in top_values]

Path("../results/00").mkdir(parents=True, exist_ok=True)
explanations = fetch_neuronpedia_explanations(
    model_id,
    sae_id,
    feature_ids,
    activations,
    save_path="../results/00/top_features.csv",
)

df = pd.DataFrame(explanations)
display(df)

  [1/20] Feature 208 sparsity=7.77%
  [2/20] Feature 667 sparsity=1.39%
  [3/20] Feature 210 sparsity=3.43%
  [4/20] Feature 7055 sparsity=0.30%
  [5/20] Feature 374 sparsity=9.49%
  [6/20] Feature 1553 sparsity=0.37%
  [7/20] Feature 249 sparsity=8.45%
  [8/20] Feature 63 sparsity=8.32%
  [9/20] Feature 485 sparsity=16.75%
  [10/20] Feature 90 sparsity=2.01%
  [11/20] Feature 6639 sparsity=0.51%
  [12/20] Feature 1732 sparsity=2.46%
  [13/20] Feature 613 sparsity=4.52%
  [14/20] Feature 1961 sparsity=0.47%
  [15/20] Feature 1110 sparsity=1.24%
  [16/20] Feature 6530 sparsity=0.12%
  [17/20] Feature 272 sparsity=5.82%
  [18/20] Feature 635 sparsity=1.06%
  [19/20] Feature 710 sparsity=1.92%
  [20/20] Feature 81 sparsity=10.31%
Saved to '..\results\00\top_features.csv'


,Rank,Feature ID,Activation,Explanation,Top Positive Tokens,Sparsity,Neuronpedia URL
0,1,208,289.7500,"Himalayas, particularly in the region around; ...","sogenannte, sogenannten, southern, 稱為, northea...",7.77,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
1,2,667,283.7500,"Statc of Punjab\nVers; ""The Joke of the Centur...","sorts, 》、《, mankind, sirens, mencionado, kaca,...",1.39,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
2,3,210,266.2500,seat of the German government and the; * **Uni...,"proclamation, proclam, gouver, treaties, terri...",3.43,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
3,4,7055,210.2500,capital of Argentina is **Buenos Aires; of Bul...,"প্তি, शहर, हट, 潴, هە, měst, நகரம், городов",0.30,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
4,5,374,199.0000,user\nlist of Jamaican politicians; provide a ...,"</h2>, </h1>, )?, </th>, </h3>, essay, ?",9.49,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
5,6,1553,181.7500,assigned to the variable `today`.; without usi...,"H, 《, An, টি, trillion, Ar, extraordinaire, be...",0.37,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
6,7,249,151.1250,Substance Abuse and Mental Health Services Adm...,"humanitarian, ciencias, sciences, bienestar, i...",8.45,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
7,8,63,139.6250,. ▁**Mount Elbrus**; Venkatarama Kulkarni; fil...,"N, P, M, G, B, Man, C, E",8.32,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
8,9,485,120.0000,"me – what are your initial thoughts; , could y...","<unused2169>, <unused2197>, <unused2204>, <unu...",16.75,https://www.neuronpedia.org/gemma-3-1b-it/13-g...
9,10,90,115.5625,"right here in Newark, Delaware!; downtown St. ...","hosted, resident, skyline, municipality, vicin...",2.01,https://www.neuronpedia.org/gemma-3-1b-it/13-g...


## D. SAE Reconstruction Quality

Measure how well the SAE reconstructs the original residual stream. The SAE input (original residual) is compared to the SAE output (reconstruction) using MSE and $\text{R}^2$.

**Note:** The fp16 values are cast to float32 before computing metrics, as fp16 squared values overflow the maximum.

In [11]:
sae_input_hook = model.get_sae_hook_name(sae, internal="hook_sae_input")
sae_output_hook = model.get_sae_hook_name(sae, internal="hook_sae_output")

resid_original = cache[sae_input_hook].float()
resid_reconstructed = cache[sae_output_hook].float()

error = resid_original - resid_reconstructed
mse_per_pos = (error**2).mean(dim=-1)
variance_per_pos = (
    (resid_original - resid_original.mean(dim=-1, keepdim=True)) ** 2
).mean(dim=-1)

r_squared = 1 - mse_per_pos / variance_per_pos

print("Reconstruction quality (per token position):")
for position in range(tokens.shape[1]):
    token_string = model.tokenizer.decode(tokens[0, position])
    print(
        f"  Position {position} ('{token_string}'): R^2 = {r_squared[0, position].item():.4f}, MSE = {mse_per_pos[0, position].item():.4f}"
    )

overall_mse = mse_per_pos.mean().item()
overall_r2 = r_squared.mean().item()
print(f"\nOverall: R^2 = {overall_r2:.4f}, MSE = {overall_mse:.4f}")

Reconstruction quality (per token position):
  Position 0 ('<bos>'): R^2 = 0.9998, MSE = 457.1217
  Position 1 ('The'): R^2 = 0.9787, MSE = 536.0424
  Position 2 (' Eiffel'): R^2 = 0.9901, MSE = 418.7332
  Position 3 (' Tower'): R^2 = 0.9900, MSE = 438.9252
  Position 4 (' is'): R^2 = 0.9926, MSE = 267.9877
  Position 5 (' in'): R^2 = 0.9886, MSE = 290.8344
  Position 6 (' the'): R^2 = 0.9889, MSE = 315.7792
  Position 7 (' city'): R^2 = 0.9889, MSE = 331.0110
  Position 8 (' of'): R^2 = 0.9879, MSE = 327.3823

Overall: R^2 = 0.9895, MSE = 375.9797


In [12]:
Path("../results/00").mkdir(parents=True, exist_ok=True)
metrics = {
    "prompt": prompt,
    "layer": layer,
    "n_active_features": n_active,
    "overall_r_squared": overall_r2,
    "overall_mse": overall_mse,
    "top_prediction": model.tokenizer.decode(top_token),
}
with open("../results/00/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to results/00/metrics.json")

Metrics saved to results/00/metrics.json


## E. Batch Verification Across Multiple Prompts

10 different factual prompts are ran to verify that the SAE produces interpretable features. Check that the model predicts the correct next token and that L0 sparsity is reasonable.

In [15]:
test_prompts = [
    "The Eiffel Tower is in the city of",
    "The capital of Japan is",
    "Water freezes at",
    "The largest planet in our solar system is",
    "Python is a programming",
    "Shakespeare wrote Romeo and",
    "The speed of light is approximately",
    "Photosynthesis converts sunlight into",
    "The Great Wall of China is located in",
    "The chemical symbol for gold is",
]

results = []
for prompt in test_prompts:
    tokens = model.to_tokens(prompt)
    logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])

    sae_acts = cache[hook_name][0, -1, :]
    top_value, top_index = sae_acts.topk(5)

    top_prediction = model.tokenizer.decode(logits[0, -1].argmax())
    n_active = (sae_acts > 0).sum().item()

    results.append(
        {
            "prompt": prompt,
            "top_prediction": top_prediction,
            "L0": n_active,
            "top_5_features": [index.item() for index in top_index],
            "top_5_activations": [value.item() for value in top_value],
        }
    )
    print(f"'{prompt}' -> '{top_prediction}' (L0 = {n_active})")

with open("../results/00/batch_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("\nBatch metrics saved to 'results/00/batch_metrics.json'")

'The Eiffel Tower is in the city of' -> ' Paris' (L0 = 70)
'The capital of Japan is' -> ' Tokyo' (L0 = 73)
'Water freezes at' -> ' ' (L0 = 70)
'The largest planet in our solar system is' -> ' Jupiter' (L0 = 69)
'Python is a programming' -> ' language' (L0 = 64)
'Shakespeare wrote Romeo and' -> ' Juliet' (L0 = 53)
'The speed of light is approximately' -> ' ' (L0 = 49)
'Photosynthesis converts sunlight into' -> ' chemical' (L0 = 58)
'The Great Wall of China is located in' -> ' the' (L0 = 51)
'The chemical symbol for gold is' -> ' "' (L0 = 58)

Batch metrics saved to 'results/00/batch_metrics.json'


## F. fp16 Stability Check

Gemma 3 was trained in bf16 but here it was loaded in fp16 (old GPU 😅). Check for potential NaN/Inf issues in logits and SAE activations across prompts of varying lengths.

In [16]:
stability_prompts = [
    "The",
    "A very long sentence with many words to test whether the model produces NaN values "
    * 3,
    "1 + 1 =",
]

instability = False
for prompt in stability_prompts:
    tokens = model.to_tokens(prompt)
    if tokens.shape[1] > config["model"]["max_seq_len"]:
        tokens = tokens[:, : config["model"]["max_seq_len"]]

    logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])
    has_nan = torch.isnan(logits).any().item()
    has_inf = torch.isinf(logits).any().item()
    sae_acts = cache[hook_name]
    sae_nan = torch.isnan(sae_acts).any().item()

    print(
        f"Prompt length={tokens.shape[1]}: logit_nan={has_nan}, logit_inf={has_inf}, sae_nan={sae_nan}"
    )
    if has_nan or has_inf or sae_nan:
        print("WARNING: Numerical instability detected!")
        instability = True

if not instability:
    print("\nAll stability checks passed.")

Prompt length=2: logit_nan=False, logit_inf=False, sae_nan=False
Prompt length=47: logit_nan=False, logit_inf=False, sae_nan=False
Prompt length=6: logit_nan=False, logit_inf=False, sae_nan=False

All stability checks passed.
